# Webscraping News – Data Wrangling & Analyse

**Ziel:** Alle webgescrapten Finanznachrichten zusammenführen, bereinigen und für die Sentimentanalyse vorbereiten.

**Datenquellen:**
- RSS-Feeds: MarketWatch, FXStreet, CNBC, Investing.com (blockiert)
- Reddit: r/Forex, r/investing, r/economics

**Schritte:**
1. Daten laden und zusammenführen (Merge)
2. Explorative Datenanalyse (EDA)
3. Duplikaterkennung und -behandlung
4. Lückenanalyse (Missing Values)
5. Spaltenreduktion für Sentimentanalyse
6. Bereinigte Daten speichern

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use("seaborn-v0_8")

# Pfade relativ zum Notebook
RAW_DIR = os.path.join("..", "..", "data", "raw", "news", "webscraping")
PROCESSED_DIR = os.path.join("..", "..", "data", "processed", "news")

# Ausgabeverzeichnis erstellen falls nötig
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Rohdaten-Verzeichnis: {os.path.abspath(RAW_DIR)}")
print(f"Ausgabe-Verzeichnis:  {os.path.abspath(PROCESSED_DIR)}")

## 2. Daten laden & zusammenführen (Merge)

Alle `all_scraped_news_*.csv` Dateien werden eingelesen und zu einem einzigen DataFrame zusammengeführt. Diese Dateien enthalten bereits die kombinierten RSS- und Reddit-Daten pro Scraping-Durchlauf.

**Wichtig:** Dieser Schritt ist jederzeit wiederholbar – es werden immer alle verfügbaren CSV-Dateien neu eingelesen.

In [ ]:
# Alle all_scraped_news CSV-Dateien finden und laden
csv_pattern = os.path.join(RAW_DIR, "all_scraped_news_*.csv")
csv_files = sorted(glob.glob(csv_pattern))

print(f"Gefundene Dateien ({len(csv_files)}):")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")

# Alle Dateien einlesen und zusammenführen
dfs = []
for f in csv_files:
    df_tmp = pd.read_csv(f)
    df_tmp["scrape_file"] = os.path.basename(f)  # Herkunft merken
    dfs.append(df_tmp)
    print(f"  {os.path.basename(f)}: {len(df_tmp)} Zeilen, {df_tmp.columns.tolist()}")

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nGesamtdatensatz: {df_raw.shape[0]} Zeilen, {df_raw.shape[1]} Spalten")

## 3. Explorative Datenanalyse (EDA)

Gemäss der EDA-Methodik untersuchen wir zunächst:
- **Shape & Datentypen** – Dimensionen und Spaltentypen
- **Erste Einblicke** – Stichprobe der Daten
- **Deskriptive Statistik** – Übersicht der numerischen und kategorialen Spalten
- **Quellenverteilung** – Wie viele Artikel pro Quelle?

In [ ]:
# 3.1 Shape & Datentypen
print("=== Shape ===")
print(f"Zeilen: {df_raw.shape[0]}, Spalten: {df_raw.shape[1]}\n")

print("=== Datentypen ===")
print(df_raw.dtypes)
print()

print("=== Erste 5 Zeilen ===")
df_raw.head()

In [ ]:
# 3.2 Deskriptive Statistik
print("=== Deskriptive Statistik (numerisch) ===")
display(df_raw.describe())

print("\n=== Deskriptive Statistik (kategorial) ===")
display(df_raw.describe(include="object"))

In [ ]:
# 3.3 Quellenverteilung
print("=== Artikel pro Quelle ===")
source_counts = df_raw["source"].value_counts()
print(source_counts)

fig, ax = plt.subplots(figsize=(10, 5))
source_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Anzahl Artikel")
ax.set_ylabel("Quelle")
ax.set_title("Verteilung der Artikel nach Quelle")
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Zeitliche Verteilung der Artikel
df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw["date_only"] = pd.to_datetime(df_raw["date_only"], errors="coerce")

articles_per_day = df_raw.groupby("date_only").size()

fig, ax = plt.subplots(figsize=(12, 4))
articles_per_day.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Datum")
ax.set_ylabel("Anzahl Artikel")
ax.set_title("Anzahl Artikel pro Tag")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"\nZeitraum: {df_raw['date_only'].min()} bis {df_raw['date_only'].max()}")
print(f"Anzahl verschiedene Tage: {df_raw['date_only'].nunique()}")

## 4. Duplikaterkennung & -behandlung

Duplikate können entstehen, weil:
- Mehrere Scraping-Durchläufe dieselben Artikel erfassen (zeitliche Überlappung)
- RSS-Feeds und Reddit dieselben Nachrichten referenzieren

**Strategie:** Wir identifizieren Duplikate basierend auf `title` und `link`, da diese eindeutig einen Artikel identifizieren.

In [ ]:
# 4.1 Exakte Duplikate (alle Spalten identisch)
exact_dupes = df_raw.duplicated().sum()
print(f"Exakte Duplikate (alle Spalten): {exact_dupes}")

# 4.2 Duplikate basierend auf title + link
title_link_dupes = df_raw.duplicated(subset=["title", "link"]).sum()
print(f"Duplikate (title + link):        {title_link_dupes}")

# 4.3 Duplikate nur basierend auf title
title_dupes = df_raw.duplicated(subset=["title"]).sum()
print(f"Duplikate (nur title):           {title_dupes}")

# 4.4 Duplikate nur basierend auf link
link_dupes = df_raw.duplicated(subset=["link"]).sum()
print(f"Duplikate (nur link):            {link_dupes}")

In [ ]:
# 4.5 Beispiele von Duplikaten anzeigen (basierend auf title + link)
dupe_mask = df_raw.duplicated(subset=["title", "link"], keep=False)
if dupe_mask.any():
    print(f"Beispiele von doppelten Einträgen ({dupe_mask.sum()} betroffene Zeilen):\n")
    dupes_sample = df_raw[dupe_mask].sort_values("title").head(10)
    display(dupes_sample[["date_only", "source", "title", "scrape_file"]])
else:
    print("Keine Duplikate gefunden.")

In [ ]:
# 4.6 Duplikate entfernen – behalte den ersten Eintrag (ältester Scrape)
df_raw_sorted = df_raw.sort_values("date")
df_deduped = df_raw_sorted.drop_duplicates(subset=["title", "link"], keep="first").copy()

removed = len(df_raw) - len(df_deduped)
print(f"Vor Deduplizierung:  {len(df_raw)} Zeilen")
print(f"Entfernte Duplikate: {removed}")
print(f"Nach Deduplizierung: {len(df_deduped)} Zeilen")

## 5. Lückenanalyse (Missing Values)

Wir untersuchen fehlende Werte pro Spalte und visualisieren deren Verteilung. Gemäss der Vorlesung unterscheiden wir:
- **MCAR** (Missing Completely At Random) – Fehlend ohne Muster
- **MAR** (Missing At Random) – Fehlend abhängig von anderen beobachteten Variablen
- **MNAR** (Missing Not At Random) – Fehlend abhängig vom fehlenden Wert selbst

Bei Textnachrichten sind fehlende `summary`-Felder typischerweise **MAR** – sie hängen von der Quelle ab (z.B. Reddit-Posts haben oft keine Summary).

In [ ]:
# 5.1 Fehlende Werte pro Spalte
missing = df_deduped.isnull().sum()
missing_pct = (df_deduped.isnull().sum() / len(df_deduped) * 100).round(2)

missing_report = pd.DataFrame({
    "Fehlend": missing,
    "Prozent (%)": missing_pct
})
print("=== Fehlende Werte pro Spalte ===")
display(missing_report)

In [ ]:
# 5.2 Fehlende Werte nach Quelle aufschlüsseln
print("=== Fehlende Werte pro Quelle ===\n")
for col in ["title", "summary", "published", "link"]:
    missing_by_source = df_deduped.groupby("source")[col].apply(lambda x: x.isnull().sum())
    if missing_by_source.sum() > 0:
        print(f"Spalte '{col}':")
        print(missing_by_source[missing_by_source > 0])
        print()

In [ ]:
# 5.3 Visualisierung fehlender Werte
fig, ax = plt.subplots(figsize=(10, 4))
missing_pct_nonzero = missing_pct[missing_pct > 0].sort_values(ascending=False)

if len(missing_pct_nonzero) > 0:
    missing_pct_nonzero.plot(kind="barh", ax=ax, color="coral")
    ax.set_xlabel("Fehlend (%)")
    ax.set_title("Anteil fehlender Werte pro Spalte")
    plt.tight_layout()
    plt.show()
else:
    print("Keine fehlenden Werte vorhanden.")

In [ ]:
# 5.4 Zeitliche Lücken analysieren
all_dates = pd.date_range(
    start=df_deduped["date_only"].min(),
    end=df_deduped["date_only"].max(),
    freq="D"
)
covered_dates = df_deduped["date_only"].dt.normalize().unique()
missing_dates = all_dates.difference(covered_dates)

print(f"Zeitraum:            {all_dates.min().date()} bis {all_dates.max().date()}")
print(f"Tage mit Daten:      {len(covered_dates)}")
print(f"Tage ohne Daten:     {len(missing_dates)}")

if len(missing_dates) > 0:
    print(f"\nFehlende Tage:")
    for d in missing_dates:
        print(f"  - {d.date()}")

## 6. Datenbereinigung & Spaltenreduktion

Für die Sentimentanalyse benötigen wir nur:
- **`date`** – Zeitstempel des Artikels
- **`date_only`** – Datum ohne Uhrzeit (für Aggregation)
- **`source`** – Herkunft (RSS-Feed oder Reddit-Subreddit)
- **`title`** – Titel des Artikels/Posts
- **`summary`** – Zusammenfassung/Inhalt

Spalten wie `link`, `published`, `score`, `num_comments`, `upvote_ratio` und `scrape_file` werden entfernt, da sie für die Sentimentanalyse nicht relevant sind.

**Behandlung fehlender Textfelder:**
- Zeilen ohne `title` UND ohne `summary` werden entfernt (kein Text für Sentiment)
- Bei fehlendem `summary` wird der `title` als Textgrundlage verwendet

In [ ]:
# 6.1 Spalten auswählen
cols_to_keep = ["date", "date_only", "source", "title", "summary"]
df_clean = df_deduped[cols_to_keep].copy()

print(f"Spalten vorher: {df_deduped.columns.tolist()}")
print(f"Spalten nachher: {df_clean.columns.tolist()}")
print(f"Zeilen: {len(df_clean)}")

In [ ]:
# 6.2 Fehlende Textfelder behandeln
# Zeilen ohne title UND ohne summary entfernen
no_text = df_clean["title"].isnull() & df_clean["summary"].isnull()
print(f"Zeilen ohne jeglichen Text (title + summary fehlen): {no_text.sum()}")
df_clean = df_clean[~no_text].copy()

# Kombinierten Text erstellen für Sentimentanalyse
# Wenn summary vorhanden -> summary verwenden, sonst title
df_clean["text"] = df_clean["summary"].fillna(df_clean["title"])

print(f"\nNach Bereinigung: {len(df_clean)} Zeilen")
print(f"Leere 'text'-Felder: {df_clean['text'].isnull().sum()}")
df_clean.head()

## 7. Qualitätsprüfung (Data Quality Check)

Abschliessende Prüfung des bereinigten Datensatzes:
- Keine Duplikate mehr vorhanden?
- Keine fehlenden Pflichtfelder?
- Datentypen korrekt?

In [ ]:
# 7.1 Qualitätsprüfung
print("=== Qualitätsprüfung ===\n")

# Duplikate
remaining_dupes = df_clean.duplicated(subset=["title"]).sum()
print(f"Verbleibende Duplikate (title): {remaining_dupes}")

# Fehlende Werte
print(f"\nFehlende Werte:")
print(df_clean.isnull().sum())

# Datentypen
print(f"\nDatentypen:")
print(df_clean.dtypes)

# Shape
print(f"\nFinaler Datensatz: {df_clean.shape[0]} Zeilen, {df_clean.shape[1]} Spalten")

# Zusammenfassung
print(f"\n=== Zusammenfassung der Datenbereinigung ===")
print(f"  Rohdaten (alle CSVs):     {len(df_raw)} Zeilen")
print(f"  Nach Deduplizierung:      {len(df_deduped)} Zeilen (-{len(df_raw) - len(df_deduped)})")
print(f"  Nach Textbereinigung:     {len(df_clean)} Zeilen (-{len(df_deduped) - len(df_clean)})")
print(f"  Reduktionsrate:           {((1 - len(df_clean)/len(df_raw)) * 100):.1f}%")

## 8. Bereinigte Daten speichern

Der bereinigte Datensatz wird als CSV in `data/processed/news/` gespeichert.

In [ ]:
# 8.1 Bereinigte Daten speichern
output_file = os.path.join(PROCESSED_DIR, "webscraping_news_cleaned.csv")
df_clean.to_csv(output_file, index=False)
print(f"Bereinigte Daten gespeichert: {output_file}")
print(f"Dateigrösse: {os.path.getsize(output_file) / 1024:.1f} KB")
print(f"Zeilen: {len(df_clean)}, Spalten: {df_clean.shape[1]}")

## Fazit

**Durchgeführte Data-Wrangling-Schritte:**

| Schritt | Methode | Ergebnis |
|---------|---------|----------|
| Datenintegration | `pd.concat()` über alle Scraping-Dateien | Zusammenführung aller CSVs |
| Duplikaterkennung | `duplicated(subset=["title", "link"])` | Identifikation und Entfernung |
| Lückenanalyse | `isnull()`, zeitliche Abdeckung | Missing Values & fehlende Tage |
| Spaltenreduktion | Auswahl relevanter Spalten | Nur Datum, Quelle, Text |
| Textkonsolidierung | `summary.fillna(title)` | Einheitliches `text`-Feld |
| Qualitätsprüfung | Validierung aller Schritte | Sauberer Datensatz |

**Bekannte Einschränkungen:**
- RSS/Reddit liefern nur aktuelle Daten, keine historischen Nachrichten für den Studienzeitraum 2024-2025
- Investing.com RSS ist blockiert (HTTP 403)
- Zeitliche Abdeckung hängt von Scraping-Häufigkeit ab